<a href="https://colab.research.google.com/github/taniaharsono/dissertation/blob/main/Copy_of_Export_Query_Results_to_PyTorch_Geometric_Node_Property_Prediction_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overview and Goals
In this notebook, we demonstrate a basic usage of exporting query results from Kùzu to PyG using the `get_as_torch_geometric()` function on the query results. In the scenario, we extract a graph from Kùzu to PyG, train a graph convolutional network (GCN) model, make predictions on some nodes whose properties are missing; and finally save those back to Kùzu, so they can be further queried. `get_as_torch_geometric()` [returns 4 values](https://kuzudb.com/docs/client-apis/python-api/query-result.html#query_result.QueryResult.get_as_torch_geometric), the goals of the demo are to show how to use the first two of these return values:
1. `pyg_data`: A [torch_geometric.data](https://pytorch-geometric.readthedocs.io/en/latest/modules/data.html) object, specifically a `Data` or `HeteroData` object, that corresponds to the nodes and edges in the query result that is being converted and auto-converted properties of those nodes as tensors. See the [get_as_torch_geometric()](https://kuzudb.com/docs/client-apis/python-api/query-result.html#query_result.QueryResult.get_as_torch_geometric) documentation for the rules of which node properties are converted to what type of tensors.
2. `pos_to_pk_map`: A map that maps the positional offset of each node `u` in `pyg_data` to the primary key of `u` in the Kùzu database.

We first describe the dataset and then the demonstration scenario.

## Cora Dataset
We will be working on the popular citation network dataset "Cora" from the “Revisiting Semi-Supervised Learning with Graph Embeddings” paper. This is one of the [benchmark datasets in PyG](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.Planetoid.html#torch-geometric-datasets-planetoid). We will simply refer to it as the "Cora dataset". The dataset consists of 2708 machine learning papers as nodes and 10556 citation relationships between them. The papers have two attributes in the original csv files, which we'll import to Kùzu as node properties (citation relationships don't have attributes).
*   title_bools: A 1433-size boolean array that describes the words included in each paper's title. If a paper p's title contains word `i`, `p.title_bools[i]` is "true". Otherwise it is false.
*   category: A numeric value between 0-6 to indicate the category of a each paper in numeric form. You can think of the categories as one of 7 possible values ("Neural Networks", "Case Based", "Reinforcement Learning", etc.) but stored as a number.

## Demonstration Scenario
In the scenario, we will import the the Cora dataset into Kùzu in a natural way: a set of `paper` nodes and `cites` relationships between papers from original csv files. We have set the categories of 5 papers as NULL for demonstration purposes. The high-level steps of the scenario are:
1. Import paper/citation.csv files into Kùzu.
2. Train PyG GCN using paper nodes with non-NULL categories. The GCN will use the title to predict the category.
3. Predict the categories of paper nodes with NULL categories.
4. Store these predictions back to Kùzu.

### Step 0: Preliminaries
Necessary package installations and imports.

In [ ]:
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git
!pip install kuzu==0.0.7

2.1.0+cu121
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Wget dataset files.

In [ ]:
!rm -rf *.csv
!wget https://kuzudb.com/data/cora/paper.csv
!wget https://kuzudb.com/data/cora/cites.csv
# Let us print the first few lines of paper.csv to see
# how the original paper records are stored in csv files.
import pandas as pd
df = pd.read_csv("./paper.csv")
df.head()

--2024-01-25 19:08:51--  https://kuzudb.com/data/cora/paper.csv
Resolving kuzudb.com (kuzudb.com)... 172.67.193.96, 104.21.76.111, 2606:4700:3033::6815:4c6f, ...
Connecting to kuzudb.com (kuzudb.com)|172.67.193.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/csv]
Saving to: ‘paper.csv’

paper.csv               [  <=>               ]  22.18M  85.9MB/s    in 0.3s    

2024-01-25 19:08:52 (85.9 MB/s) - ‘paper.csv’ saved [23262865]

--2024-01-25 19:08:52--  https://kuzudb.com/data/cora/cites.csv
Resolving kuzudb.com (kuzudb.com)... 172.67.193.96, 104.21.76.111, 2606:4700:3033::6815:4c6f, ...
Connecting to kuzudb.com (kuzudb.com)|172.67.193.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/csv]
Saving to: ‘cites.csv’

cites.csv               [ <=>                ]  94.61K  --.-KB/s    in 0.02s   

2024-01-25 19:08:52 (5.86 MB/s) - ‘cites.csv’ saved [96880]



,id,title_bools,category
0,0,"[false,false,false,false,false,false,false,fal...",3.0
1,1,"[false,false,false,false,false,false,false,fal...",4.0
2,2,"[false,false,false,false,false,false,false,fal...",4.0
3,3,"[false,false,false,false,false,false,false,fal...",0.0
4,4,"[false,false,false,true,false,false,false,fals...",3.0


### Step 1: Import the CSV Files Into Kùzu
We create the schemas for `paper` nodes and `cites` relationships in Kùzu and then import the csv files into Kùzu using COPY FROM clause.

In [ ]:
import kuzu as kz
import shutil
shutil.rmtree("./cora", ignore_errors=True)
db = kz.Database("./cora")
conn = kz.Connection(db)

In [ ]:
conn.execute("CREATE NODE TABLE paper (id INT64, title_bools BOOLEAN[], category INT64, PRIMARY KEY (id))")
conn.execute("CREATE REL table cites (FROM paper TO paper, MANY_MANY)")
conn.execute('COPY paper FROM "./paper.csv" (HEADER=true)')
conn.execute('COPY cites FROM "./cites.csv" (HEADER=true)')
# Let us print the first few paper records to see
# how they look once they are imported into Kùzu as nodes.
paper_nodes_df = conn.execute("MATCH (a:paper) RETURN a LIMIT 5").get_as_df();
paper_nodes_df.head()

,a
0,"{'_label': 'paper', '_id': {'offset': 2048, 't..."
1,"{'_label': 'paper', '_id': {'offset': 2049, 't..."
2,"{'_label': 'paper', '_id': {'offset': 2050, 't..."
3,"{'_label': 'paper', '_id': {'offset': 2051, 't..."
4,"{'_label': 'paper', '_id': {'offset': 2052, 't..."


We can also verify that some of the paper nodes have NULL categories and print them.
  

In [ ]:
null_category_papers = conn.execute("MATCH (a:paper) WHERE a.category IS NULL RETURN a").get_as_df();
null_category_papers

,a
0,"{'_label': 'paper', '_id': {'offset': 2703, 't..."
1,"{'_label': 'paper', '_id': {'offset': 2704, 't..."
2,"{'_label': 'paper', '_id': {'offset': 2705, 't..."
3,"{'_label': 'paper', '_id': {'offset': 2706, 't..."
4,"{'_label': 'paper', '_id': {'offset': 2707, 't..."


### Step 2: Train PyG GCN Using Paper Nodes With non-NULL Categories
We next get all paper nodes that have non-NULL categories and the citations between them and covert to a PyG `torch_geometric.data`. The first return value of `get_as_torch_geometric()` converts a query result into a `torch_geometric.data` object. In the below line, this is the `train_data` variable. This returned object can either be a [`Data` or `HeteroData` object](https://pytorch-geometric.readthedocs.io/en/latest/modules/data.html) and will contain auto-converted node attributes. Overall, query results that contain a single node and relationship label are converted to `Data` objects and multiple node and/or relationship labels are converted to `HeteroData` objects. For auto-conversion:  any node attribute that is numeric or boolean (or lists of numeric or booleans) are converted as node features in the PyG `torch_geometric.data` objects though there are several reasons they may not be. See the [Kùzu documentation](https://) for details on the rules of how the `get_as_torch_geometric()` function converts query results into `torch_geometric.data` objects and auto-converts node attributes.

Below `train_data` is a PyG `Data` object because there is only one type of node label (`paper`) and relationship label (`cites`) in the query results. For now, this is all we need in this scenario to train a GCN model to predict categories using `title_bool` attributes of nodes.

In [ ]:
train_data, _, _, _ = conn.execute("MATCH (a:paper)-[r:cites]->(b:paper) WHERE a.category IS NOT NULL and b.category IS NOT NULL RETURN *").get_as_torch_geometric()

We will be using the GCN model defined by the PyG team in this Colab, so majority of the code below that is related to the actual training of the GCN model is copy-pasted from [this colab notebook](https://colab.research.google.com/drive/14OvFnAXggxB8vM4e8vSURUp1TaKnovzX?usp=sharing) that the PyG team has authored.

The GCN model here requires the `title_bools` to be a tensor of floats, so we first convert `title_bools`, which is a tensor of booleans to tensor of floats. We then assign the tensor `train_data.x` to be `title_bools` and `train_data.y` to `category` (you can see the `torch_geometric.data.Data`'s documentation for the `x` and `y` tensors of `Data` instances).

In [ ]:
train_data.title_bools = train_data.title_bools.float()
train_data.x = train_data.title_bools
train_data.y = train_data.category

We next prepare the train-test splits as a 60%-40% split and and prepare the necessary train and test masks.

In [ ]:
total_count = len(train_data.title_bools)
train_count = int(0.6 * total_count)
test_count = int(0.4 * total_count)
train_ids, test_ids = torch.utils.data.random_split(
    range(total_count), (train_count, test_count),
    generator=torch.Generator().manual_seed(42)
)
train_mask = torch.zeros(total_count, dtype=torch.bool)
test_mask = torch.zeros(total_count, dtype=torch.bool)
train_mask.index_fill_(0, torch.LongTensor(train_ids), True)
test_mask.index_fill_(0, torch.LongTensor(train_ids), True)
train_data.train_mask = train_mask
train_data.test_mask = test_mask
train_data

Data(id=[2700], title_bools=[2700, 1433], category=[2700], edge_index=[2, 10536], x=[2700, 1433], y=[2700], train_mask=[2700], test_mask=[2700])

Next we define the GCN model. This is directly copy-pasted from the [PyG team's notebook](https://colab.research.google.com/drive/14OvFnAXggxB8vM4e8vSURUp1TaKnovzX?usp=sharing).

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
class GCN(torch.nn.Module):
    def __init__(self, input_channels, hidden_channels, output_channels):
        super().__init__()
        torch.manual_seed(42)
        self.conv1 = GCNConv(input_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, output_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model = GCN(input_channels=train_data.x.shape[1], hidden_channels=32, output_channels=train_data.category.max().item() + 1)
print(model)

GCN(
  (conv1): GCNConv(1433, 32)
  (conv2): GCNConv(32, 7)
)


Next we train the model. Again, this is directly copy-pasted from the [PyG team's notebook](https://colab.research.google.com/drive/14OvFnAXggxB8vM4e8vSURUp1TaKnovzX?usp=sharing). This completes Step 2 of the demo.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

def train(data):
      model.train()
      optimizer.zero_grad()
      out = model(data.title_bools, data.edge_index)
      loss = criterion(out[data.train_mask], data.category[data.train_mask])
      loss.backward()
      optimizer.step()
      return loss

def pred(data):
      model.eval()
      out = model(data.title_bools, data.edge_index)
      pred = out.argmax(dim=1)
      return pred

def test(data):
      predictions = pred(data)
      test_correct = predictions[data.test_mask] == data.y[data.test_mask]
      test_acc = int(test_correct.sum()) / int(data.test_mask.sum())
      return test_acc

for epoch in range(1, 101):
    loss = train(train_data)
    test_acc = test(train_data)
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Test Acc: {test_acc:.4f}')

Epoch: 001, Loss: 1.9506, Test Acc: 0.3333
Epoch: 002, Loss: 1.7732, Test Acc: 0.3451
Epoch: 003, Loss: 1.6050, Test Acc: 0.3981
Epoch: 004, Loss: 1.4631, Test Acc: 0.5698
Epoch: 005, Loss: 1.3131, Test Acc: 0.7154
Epoch: 006, Loss: 1.1672, Test Acc: 0.7710
Epoch: 007, Loss: 1.0409, Test Acc: 0.7907
Epoch: 008, Loss: 0.9252, Test Acc: 0.8191
Epoch: 009, Loss: 0.8331, Test Acc: 0.8395
Epoch: 010, Loss: 0.7576, Test Acc: 0.8556
Epoch: 011, Loss: 0.6638, Test Acc: 0.8716
Epoch: 012, Loss: 0.6203, Test Acc: 0.8815
Epoch: 013, Loss: 0.5601, Test Acc: 0.8914
Epoch: 014, Loss: 0.5192, Test Acc: 0.9062
Epoch: 015, Loss: 0.4708, Test Acc: 0.9136
Epoch: 016, Loss: 0.4445, Test Acc: 0.9191
Epoch: 017, Loss: 0.4192, Test Acc: 0.9259
Epoch: 018, Loss: 0.3837, Test Acc: 0.9278
Epoch: 019, Loss: 0.3607, Test Acc: 0.9290
Epoch: 020, Loss: 0.3471, Test Acc: 0.9321
Epoch: 021, Loss: 0.3268, Test Acc: 0.9346
Epoch: 022, Loss: 0.3192, Test Acc: 0.9358
Epoch: 023, Loss: 0.2912, Test Acc: 0.9364
Epoch: 024,

### Step 3: Predict the Categories of Paper Nodes With NULL Categories.

Although we only want to predict the categories of nodes whose category values are NULL, for GCNs, it is simpler to get the entire graph and make predictions for all, so that's what we will do. We next get the all paper nodes, i.e.,  those with NULL as well as non-NULL categories, and the citations between them and convert to a PyG Data object. In this case, in fact this is the entire graph. Below we store this PyG Data object in the `pred_data` variable. As we did with the `train_data` PyG Data object above, we will convert the boolean tensor `title_bools` on `train_data` to a float tensor. Then we finally do the actual predictions by calling the `pred(...)` function we defined above. The result `pred_data` contains for each node the predicted category of the node.

Side note: for papers whose categories are already known you might want to check as an exercise that these predictions are very accurate (~98%), which is the accuracy level that this model achieves on this datasets. But for the purpose of this demo, we only care about the predictions for those papers with NULL categories.

In [ ]:
pred_data, pyg_pos_to_kz_pk, _, _ = conn.execute("MATCH (a:paper)-[r:cites]->(b:paper) RETURN *").get_as_torch_geometric()
pred_data.title_bools = pred_data.title_bools.float()
pred_data.x = pred_data.title_bools
pred_all = pred(pred_data).tolist()

/usr/local/lib/python3.10/dist-packages/kuzu/torch_geometric_result_converter.py:183: UserWarning: Property paper.category has a null value. torch_geometric does not support null values. The property is marked as unconverted.
  warnings.warn(message)


### Step 4: Store the Predictions of NULL-category Papers Back to Kùzu
As a final step, for all those nodes with NULL categories, we store their predicted categories by the model back to the database, thereby completing missing data in the database using predicted values. We do this in two steps:
1. We fetch the primary keys (pk) (which is the `id` property of `paper` nodes) of all paper nodes whose category is NULL. We store these in a `null_ids` field.
2. We next find all predictions for these nodes from the `pred_all` list, which contains predictions we computed. To do this, we need to map the positional offset of the nodes that were in the PyG Data object we created to the primary keys. This requires using the 2nd return value of `get_as_torch_geometric()` function that we used above, which we stored in the `pyg_pos_to_kz_pk` variable: the "pyg data position to kuzu primary key (pk)" map. You can use this map if you want to get the primary key for each node position in a PyG Data or HeteroData object that is created by the `get_as_torch_geometric()` function. We loop through each PyG node position `pos` in `pyg_pos_to_kz_pk` and if the primary key `paper_id` for that `pos` is in the `null_ids`, then we issue a `SET` query to Kùzu to set the `category` property of the node with this `paper_id` to the predicted category from the model.

In [ ]:
null_ids = conn.execute("MATCH (a:paper) WHERE a.category IS NULL RETURN a.id").get_as_df()
null_ids = set(null_ids['a.id'])
for pos in pyg_pos_to_kz_pk:
    paper_id = pyg_pos_to_kz_pk[pos]
    if paper_id in null_ids:
        prediction = pred_all[pos]
        conn.execute("MATCH (a:paper) WHERE a.id = $pid SET a.category = $pred", [("pid", paper_id), ("pred", prediction)])

As a verification step, we can verify that there are no paper nodes with NULL category values.


In [ ]:
res = conn.execute("MATCH (a:paper) WHERE a.category IS NULL RETURN count(*)").get_as_df();
res

,COUNT_STAR()
0,0
